# Removation ComfyUI Cloud GPU
**Instructions:**
1. Runtime → Change runtime type → T4 GPU
2. Run all cells (Runtime → Run all)
3. Wait for tunnel URL to appear in Cell 4 output
4. Copy the URL → paste in Removation Dashboard → Settings → GPU → Colab Tunnel URL
5. Leave this tab open while using the dashboard


In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
# Cell 1.5 — Optional Direct-to-Drive High-Speed Downloader
# If you don't want to upload ComfyUI models from your local computer,
# you can download them directly from the cloud straight into your Google Drive!
# This uses Google's gigabit server speed (takes minutes instead of hours).

import os

DRIVE_MODELS = '/content/drive/MyDrive/ComfyUI_Models'

# Create directories
os.makedirs(f"{DRIVE_MODELS}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_MODELS}/controlnet", exist_ok=True)
os.makedirs(f"{DRIVE_MODELS}/upscale_models", exist_ok=True)

print("⏬ Downloading SDXL Base Model directly to your Drive...")
sdxl_base = f"{DRIVE_MODELS}/checkpoints/sd_xl_base_1.0.safetensors"
if not os.path.exists(sdxl_base):
    os.system(f'wget -c -O "{sdxl_base}" "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"')
    print("✅ SDXL Base Model saved directly to Google Drive!")
else:
    print("✅ SDXL Base Model already present in your Drive.")

print("⏬ Downloading ControlNet Depth Model directly to your Drive...")
control_depth = f"{DRIVE_MODELS}/controlnet/control_depth-fp16.safetensors"
if not os.path.exists(control_depth):
    os.system(f'wget -c -O "{control_depth}" "https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_depth-fp16.safetensors"')
    print("✅ ControlNet Depth Model saved directly to Google Drive!")
else:
    print("✅ ControlNet Depth Model already present in your Drive.")

print("⏬ Downloading ControlNet Canny Model directly to your Drive...")
control_canny = f"{DRIVE_MODELS}/controlnet/control_canny-fp16.safetensors"
if not os.path.exists(control_canny):
    os.system(f'wget -c -O "{control_canny}" "https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_canny-fp16.safetensors"')
    print("✅ ControlNet Canny Model saved directly to Google Drive!")
else:
    print("✅ ControlNet Canny Model already present in your Drive.")

print("⏬ Downloading Upscale Model (RealESRGAN) directly to your Drive...")
upscale = f"{DRIVE_MODELS}/upscale_models/RealESRGAN_x4plus.pth"
if not os.path.exists(upscale):
    os.system(f'wget -c -O "{upscale}" "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"')
    print("✅ Real-ESRGAN Model saved directly to Google Drive!")
else:
    print("✅ Real-ESRGAN Model already present in your Drive.")

print("🎉 Direct cloud download complete! Your Drive is now fully prepared.")

In [ ]:
# Cell 2 — Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print('✅ GPU ready')

In [ ]:
# Cell 3 — Install ComfyUI + link models from Google Drive
import os

COMFYUI_PATH = '/content/ComfyUI'
DRIVE_MODELS = '/content/drive/MyDrive/ComfyUI_Models'

# Clone ComfyUI if not already done
if not os.path.exists(COMFYUI_PATH):
    os.system('git clone https://github.com/comfyanonymous/ComfyUI.git')
    os.system('pip install -r /content/ComfyUI/requirements.txt -q')
    print('✅ ComfyUI installed')
else:
    print('✅ ComfyUI already present')

# Symlink models from Google Drive instead of copying
MODEL_DIRS = [
    'checkpoints', 'loras', 'vae', 'embeddings',
    'controlnet', 'upscale_models'
]

for folder in MODEL_DIRS:
    src = f'{DRIVE_MODELS}/{folder}'
    dst = f'{COMFYUI_PATH}/models/{folder}'
    if os.path.exists(src):
        if os.path.exists(dst):
            os.system(f'rm -rf {dst}')
        os.system(f'ln -s {src} {dst}')
        print(f'✅ Linked {folder}')
    else:
        print(f'⚠️  {folder} not found in Drive — skipping')

# Link custom nodes
DRIVE_NODES = f'{DRIVE_MODELS}/custom_nodes'
DST_NODES   = f'{COMFYUI_PATH}/custom_nodes'
if os.path.exists(DRIVE_NODES):
    for node_folder in os.listdir(DRIVE_NODES):
        src = f'{DRIVE_NODES}/{node_folder}'
        dst = f'{DST_NODES}/{node_folder}'
        if not os.path.exists(dst):
            os.system(f'ln -s {src} {dst}')
    print('✅ Custom nodes linked')

# Install custom node requirements
os.system(f'''
  for req in {DST_NODES}/*/requirements.txt; do
    pip install -r "$req" -q 2>/dev/null || true
  done
''')
print('✅ Custom node requirements installed')

In [ ]:
# Cell 4 — Start ComfyUI + Cloudflare Tunnel
import subprocess, os, time, re

COMFYUI_PATH = '/content/ComfyUI'

# Install cloudflared
os.system(
    'wget -q https://github.com/cloudflare/cloudflared/releases/'
    'latest/download/cloudflared-linux-amd64 -O /usr/bin/cloudflared '
    '&& chmod +x /usr/bin/cloudflared'
)

# Start ComfyUI in background
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', '8188'],
    cwd=COMFYUI_PATH,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('Starting ComfyUI... (waiting 15s)')
time.sleep(15)

# Start Cloudflare tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

print('Starting tunnel...')
tunnel_url = None
for _ in range(30):
    line = tunnel_proc.stdout.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print()
    print('=' * 60)
    print('✅ ComfyUI is running on Google Colab T4 GPU!')
    print()
    print('🔗 YOUR TUNNEL URL:')
    print(f'   {tunnel_url}')
    print()
    print('👆 Copy this URL and paste it in:')
    print('   Removation Dashboard → Settings → GPU → Colab Tunnel URL')
    print('   Then set mode to AUTO or CLOUD ONLY and click Save.')
    print('=' * 60)
else:
    print('❌ Could not get tunnel URL. Check logs above.')

In [ ]:
# Cell 5 — Keep alive (prevents Colab ~90min idle timeout)
import time
print('Keeping session alive. Stop this cell to shut down.')
while True:
    print(f'Still running... {time.strftime("%H:%M:%S")}', end='\r')
    time.sleep(60)